# Week 66, Day 1: Processes and threads

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w066d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
downloads = [3, 1, 2, 2]          # seconds each download spends waiting on the network

one_at_a_time = sum(downloads)
all_at_once = max(downloads)       # if every wait can overlap
print(one_at_a_time, all_at_once)

def with_workers(times, workers):
    free_at = [0] * workers
    for t in times:
        i = free_at.index(min(free_at))   # the worker that becomes free first
        free_at[i] += t
    return max(free_at)

print(with_workers(downloads, 2))

In [ ]:
def timeline(tasks):
    "One cook switching between tasks: each task is a list of (action, minutes)."
    clock, log = 0, []
    waiting = []                                   # (time it finishes, name)
    for name, steps in tasks:
        for action, minutes in steps:
            if action == "work":
                clock += minutes                   # the cook is busy
                log.append((clock, f"{name}: worked {minutes}"))
            else:
                waiting.append((clock + minutes, name))   # waits overlap with other work
    return max([clock] + [end for end, _ in waiting]), log

total, log = timeline([("rice", [("work", 2), ("wait", 15)]),
                       ("salad", [("work", 8)]),
                       ("sauce", [("work", 3), ("wait", 10)])])
print(total, log[-1])
print("one dish after another:", 2 + 15 + 8 + 3 + 10)

In [ ]:
import copy, pickle

state = {"count": 0, "log": []}

def work(shared):
    shared["count"] += 1
    shared["log"].append("done")
    return shared["count"]

def as_thread(func, shared):
    return func(shared)                         # same objects: changes are visible

def as_process(func, shared):
    child_memory = copy.deepcopy(shared)        # a process gets its own copy
    result = func(child_memory)
    return pickle.loads(pickle.dumps(result))   # results come back serialised

print(as_process(work, state), state)
print(as_thread(work, state), state)
try:
    pickle.dumps(lambda x: x)
except Exception as e:
    print(type(e).__name__, "lambdas can't be sent to a process")

In [ ]:
def classify(cpu_ms, wait_ms):
    return "I/O-bound" if wait_ms > cpu_ms else "CPU-bound"

def suggest(kind):
    return {"I/O-bound": "threads or asyncio", "CPU-bound": "processes (or NumPy, C extensions)"}[kind]

tasks = {"fetch a web page": (2, 180), "resize a photo": (90, 5), "query a database": (1, 40),
         "compress a file": (400, 20)}
for name, (cpu, wait) in tasks.items():
    kind = classify(cpu, wait)
    print(f"{name:18} {kind:10} -> {suggest(kind)}")

In [ ]:
def speedup(p, n):
    return 1 / ((1 - p) + p / n)

for p in (0.5, 0.9, 0.99):
    print(p, [round(speedup(p, n), 1) for n in (2, 4, 16, 1024)], "limit", round(1 / (1 - p)))

def chunks(items, n):
    size, extra = divmod(len(items), n)
    out, start = [], 0
    for i in range(n):
        end = start + size + (1 if i < extra else 0)
        out.append(items[start:end])
        start = end
    return out

print([len(c) for c in chunks(list(range(10)), 4)])

> **Key idea.** Concurrency is making progress on several tasks by switching between them; parallelism is running them at the same instant on separate cores. Processes have their own memory, are isolated and relatively costly to start, and exchange copies of data (pickled, so it must be picklable); threads share their process's memory, are cheap, and are switched preemptively, so shared data needs care. I/O-bound work (mostly waiting) is helped by overlapping waits with threads or asyncio; CPU-bound work needs parallelism, which in CPython usually means processes, NumPy or C extensions. Overlapping waits cuts time from the sum towards the maximum; Amdahl's law limits the speed-up to 1 / (1 - p); overheads punish tiny tasks, so hand out sizeable chunks.

### Exercise: Choosing a model

Each task is a dict with `cpu_ms` (time computing) and `wait_ms` (time waiting). Write `kind(task)`: `"io"` if it waits longer than it computes, otherwise `"cpu"`. Then `plan(tasks)`: a dict from `"threads"` (for I/O-bound tasks) and `"processes"` (for CPU-bound ones) to the lists of task names, in order, leaving out keys with no tasks, and `best_case(tasks)`: the total `cpu_ms + wait_ms` if done one after another, and the time if every wait overlapped perfectly but computing happened on one core (the total `cpu_ms`, or the longest single task if that's larger), as a tuple.

In [ ]:
def kind(task):
    ___

def plan(tasks):
    ___

def best_case(tasks):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_classify():\n    \"I/O-bound and CPU-bound\"\n    tasks = [{\"name\": \"fetch\", \"cpu_ms\": 2, \"wait_ms\": 200}, {\"name\": \"resize\", \"cpu_ms\": 80, \"wait_ms\": 5},\n             {\"name\": \"query\", \"cpu_ms\": 1, \"wait_ms\": 50}, {\"name\": \"zip\", \"cpu_ms\": 300, \"wait_ms\": 300}]\n    assert [kind(t) for t in tasks] == [\"io\", \"cpu\", \"io\", \"cpu\"]\n    assert plan(tasks) == {\"threads\": [\"fetch\", \"query\"], \"processes\": [\"resize\", \"zip\"]}\n    assert plan(tasks[:1]) == {\"threads\": [\"fetch\"]} and plan([]) == {}\n    assert best_case(tasks) == (938, 600) and best_case(tasks[:1]) == (202, 202)")

### Exercise: Scheduling on workers

Write `makespan(durations, workers)`: give each task, in order, to the worker that becomes free **first** (the lowest-numbered worker on ties), and return the time when the last task finishes. Use a heap of `(free_at, worker)` pairs. Then `assignment(durations, workers)`: the list of which worker ran each task, and `sorted_first(durations, workers)`: the makespan if the longest tasks are handed out first (often better).

In [ ]:
import heapq

def makespan(durations, workers):
    ___

def assignment(durations, workers):
    ___

def sorted_first(durations, workers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_workers():\n    \"list scheduling\"\n    assert makespan([3, 1, 2, 2], 2) == 5 and makespan([3, 1, 2, 2], 1) == 8 and makespan([3, 1, 2, 2], 10) == 3\n    assert assignment([3, 1, 2, 2], 2) == [0, 1, 1, 0] and assignment([5, 5, 1], 3) == [0, 1, 2]\n    assert makespan([], 3) == 0\n    assert makespan([1, 1, 1, 1, 4], 2) == 6 and sorted_first([1, 1, 1, 1, 4], 2) == 4\n    assert \"heapq\" in source")

### Exercise: Amdahl's law

Write `speedup(p, n)`: Amdahl's speed-up for a parallel fraction `p` on `n` workers, rounded to 2 places; `limit(p)`: the most it can ever be, `1 / (1 - p)` rounded to 2 places (or `float("inf")` when `p` is 1); and `workers_needed(p, target)`: the smallest whole `n` whose speed-up reaches `target`, or `None` if the target is at least the limit (no number of workers is enough).

In [ ]:
def speedup(p, n):
    ___

def limit(p):
    ___

def workers_needed(p, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_amdahl():\n    \"the serial part limits the speed-up\"\n    assert speedup(0.9, 1) == 1.0 and speedup(0.9, 10) == 5.26 and speedup(0.5, 1000) == 2.0\n    assert limit(0.9) == 10.0 and limit(0.75) == 4.0 and limit(1.0) == float(\"inf\")\n    assert workers_needed(0.9, 5) == 9 and workers_needed(0.9, 10) is None and workers_needed(0.5, 1) == 1\n    assert workers_needed(1.0, 64) == 64")

### Exercise: Shared or copied?

Simulate the two memory models. `in_thread(func, shared)` calls `func(shared)` on the **same** object and returns the result. `in_process(func, shared)` gives `func` a deep **copy** (so the caller's data is never changed) and returns the result after a `pickle` round trip, as if it travelled back through a pipe; if the result can't be pickled, it raises `TypeError("result can't be sent back")`. Then `can_send(value)`: True if `pickle.dumps(value)` works.

In [ ]:
import copy
import pickle

def in_thread(func, shared):
    ___

def in_process(func, shared):
    ___

def can_send(value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_memory():\n    \"threads share, processes copy\"\n    data = {\"items\": [1, 2]}\n    def add(d):\n        d[\"items\"].append(3)\n        return len(d[\"items\"])\n    assert in_process(add, data) == 3 and data == {\"items\": [1, 2]}\n    assert in_thread(add, data) == 3 and data == {\"items\": [1, 2, 3]}\n    out = in_process(lambda d: d[\"items\"], data)\n    assert out == [1, 2, 3] and out is not data[\"items\"]\n    try:\n        in_process(lambda d: (lambda: 1), data)\n        assert False\n    except TypeError as e:\n        assert str(e) == \"result can't be sent back\"\n    assert can_send({\"a\": [1, (2, 3)]}) and not can_send(lambda: 0)\n    import threading\n    assert not can_send(threading.Lock())")

### Exercise: Dividing the work

Write `split_work(items, n)`: divide a list into `n` contiguous chunks whose sizes differ by at most one, larger chunks first, keeping order (empty chunks if there are fewer items than `n`). Then `total_in_parallel(items, n, func)`: apply `func` to each chunk (as if each went to a different worker) and add up the results, and `overhead_worth_it(work_ms, overhead_ms, n)`: True if `work_ms / n + overhead_ms * n` is less than `work_ms` (each extra worker costs a fixed start-up overhead).

In [ ]:
def split_work(items, n):
    ___

def total_in_parallel(items, n, func):
    ___

def overhead_worth_it(work_ms, overhead_ms, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chunks():\n    \"chunking\"\n    assert split_work(list(range(10)), 4) == [[0, 1, 2], [3, 4, 5], [6, 7], [8, 9]]\n    assert split_work([1, 2], 3) == [[1], [2], []] and split_work([], 2) == [[], []]\n    assert sum(len(c) for c in split_work(list(range(101)), 7)) == 101\n    assert total_in_parallel(list(range(1, 101)), 4, sum) == 5050\n    assert overhead_worth_it(1000, 10, 4) and not overhead_worth_it(20, 10, 4)")

### Bug hunt: Amdahl upside down

The formula multiplies the parallel part by `n` instead of dividing, so more workers make the program **slower**: this prints `0.11`. Divide the parallel fraction by the number of workers so it prints `5.26`.

In [ ]:
def speedup(p, n):
    return 1 / ((1 - p) + p * n)

print(round(speedup(0.9, 10), 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"Amdahl's law\"\n    expect_output(\"5.26\")")

### Bug hunt: Taking turns instead of the first free worker

The scheduler hands tasks out in strict rotation, so the second worker gets both long tasks while the first sits idle, and this prints `11`. Give each task to the worker that becomes free first so it prints `8`.

In [ ]:
durations = [1, 5, 1, 6]
free_at = [0, 0]
for i, d in enumerate(durations):
    w = i % 2
    free_at[w] += d
print(max(free_at))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"earliest free worker\"\n    expect_output(\"8\")")

### Bug hunt: Work that went missing

Each chunk gets `len(items) // n` items, so the leftover items at the end are never handed to any worker, and this prints `9`. Share out the remainder too (or let the last chunk run to the end) so it prints `10`.

In [ ]:
items = list(range(10))
n = 3
size = len(items) // n
chunks = [items[i * size:(i + 1) * size] for i in range(n)]
print(sum(len(c) for c in chunks))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no lost items\"\n    expect_output(\"10\")")

### Puzzle: The first free worker

Arrange the lines of a function that computes when a list of tasks finishes on `k` workers, giving each task to the worker free soonest. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # one core switching between tasks: "concurrency" or "parallelism"
q = ___   # True or False: two threads in one process see the same objects
r = ___   # downloading files is usually "io" or "cpu" bound
s = ___   # Amdahl's limit for p = 0.95, as a whole number
t = ___   # True or False: a lambda can be pickled and sent to another process

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"concurrency\" and q is True and r == \"io\", \"p, q and r\"\n    assert s == 20 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a timeline of tasks

Simulate tasks that alternate between computing and waiting on `cores` processor cores. Each task is `(name, steps)`, where each step is `("cpu", t)` (needs a core for `t` time units) or `("io", t)` (waits `t` units without needing a core; any number of tasks can wait at once). Write `simulate(tasks, cores)` with these rules:

- every task starts ready at time 0; ready tasks get free cores in the order they became ready (ties: the order the tasks were given);
- a task keeps its core until its CPU step finishes (no preemption), then moves on to its next step at once;
- when an I/O step finishes, the task becomes ready again (or finishes, if that was its last step).

Return a dict from each task's name to the time it finished, and write `total_time(tasks, cores)`, the time the last task finished (0 for no tasks). Process events in time order with a heap, handling everything that happens at the same time before starting new CPU work.

In [ ]:
import heapq

def simulate(tasks, cores):
    ___

def total_time(tasks, cores):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timeline():\n    \"overlapping work and waiting\"\n    io_heavy = [(\"a\", [(\"cpu\", 1), (\"io\", 5), (\"cpu\", 1)]), (\"b\", [(\"cpu\", 1), (\"io\", 5), (\"cpu\", 1)])]\n    assert simulate(io_heavy, 1) == {\"a\": 7, \"b\": 8} and total_time(io_heavy, 2) == 7\n    cpu_heavy = [(\"x\", [(\"cpu\", 4)]), (\"y\", [(\"cpu\", 4)]), (\"z\", [(\"cpu\", 4)])]\n    assert simulate(cpu_heavy, 1) == {\"x\": 4, \"y\": 8, \"z\": 12} and simulate(cpu_heavy, 3) == {\"x\": 4, \"y\": 4, \"z\": 4}\n    assert total_time(cpu_heavy, 2) == 8\n    mixed = [(\"dl\", [(\"io\", 10)]), (\"calc\", [(\"cpu\", 3), (\"cpu\", 3)])]\n    assert simulate(mixed, 1) == {\"dl\": 10, \"calc\": 6}\n    assert total_time([], 4) == 0 and simulate([(\"empty\", [])], 1) == {\"empty\": 0}")

## Warm-up

Downloading one file at a time takes the sum of the waits. If the waits can overlap, the total drops towards the longest single wait, and with two workers it lands in between. Overlapping waiting is the most common reason to use concurrency.

## Concurrency and parallelism

Two words, often confused:

- **Concurrency**: making progress on several tasks during the same period, by **switching** between them. One cook can make three dishes concurrently: start the rice, chop while it boils, stir the sauce while the oven heats. Only one thing happens at any instant, but no time is wasted waiting.
- **Parallelism**: doing several things **at the same instant**, on separate hardware. Three cooks in a kitchen work in parallel.

Concurrency is about **structure** (dealing with many things at once); parallelism is about **execution** (doing many things at once). A program can be concurrent on a single processor core; it can only be parallel with several cores. Modern machines have several cores, but most programs use just one unless they're written to do otherwise.

Concurrency costs: programs become harder to reason about, test and debug, because tasks can interleave in different orders on different runs. Use it when the benefit is clear.

## Processes and threads

The operating system runs programs as **processes**:

- A process has its **own memory**: one process can't see or change another's variables. Processes communicate only through explicit channels: pipes, sockets, files, shared memory segments.
- Starting a process is relatively **slow** (milliseconds) and each one uses its own memory, including its own Python interpreter.
- If one crashes, the others carry on: processes are **isolated**.

Inside a process, there can be several **threads**:

- Threads **share** the process's memory: every thread sees the same objects. That makes communication easy (just use the same list or dict) and dangerous (two threads changing the same thing at once can corrupt it: Thursday's race conditions).
- Starting a thread is **cheap**, and switching between threads is fast.
- The operating system decides when each thread runs (**preemptive** scheduling): a thread can be paused between any two steps.

To send data to another **process**, Python must **serialise** it, normally with `pickle`: the data is copied, never shared, and it must be picklable (lambdas, open files and locks aren't). Threads need none of that.

## I/O-bound and CPU-bound work

What a task spends its time on decides which tool helps:

- **I/O-bound** work spends most of its time **waiting**: for the network, a disk, a database, a user. The processor is idle meanwhile. Overlapping the waits gives big speed-ups, and it doesn't need more cores: **threads** (this week) or **asyncio** (next week) are the tools.
- **CPU-bound** work spends its time **computing**: image processing, simulations, compression, number crunching in pure Python. Overlapping doesn't help on one core, since there's no waiting to hide; it needs **parallelism** across cores. In CPython that usually means **processes** (`multiprocessing`, `ProcessPoolExecutor`), because of the global interpreter lock (Wednesday), or moving the work into code that runs outside Python's interpreter loop (NumPy, Week 65; C extensions, Week 68).

Profile first (Week 65): if `cumtime` is dominated by `socket.recv`, `read` or `sleep`, the work is I/O-bound; if it's in your own Python functions, it's CPU-bound.

## How much faster can it get?

Before adding concurrency, estimate the gain:

- **Overlapping waits**: n independent I/O-bound tasks with waits w₁ ... wₙ take about `sum(w)` one at a time and about `max(w)` with unlimited overlap. With k workers it's in between: a **scheduling** problem. A simple and good strategy is to give each next task to whichever worker becomes free first.
- **Amdahl's law** (Gene Amdahl, 1967): if a fraction `p` of a program can be parallelised and the rest can't, then with `n` workers the speed-up is `1 / ((1 - p) + p / n)`. Even with infinitely many workers, it can't exceed `1 / (1 - p)`: a program that's 90% parallel can never be more than 10 times faster. The serial part dominates quickly.
- **Overheads**: starting workers, sending data to processes, and coordinating all cost time. Tiny tasks can get **slower** with more workers. Give each worker a meaningful **chunk** of work.

## On your own computer

On your own machine (not in the browser), try the real thing. Save this as `downloads.py` and run it: the thread version takes about as long as the longest wait, not the sum.

```
import threading, time

def fake_download(name, seconds, results):
    time.sleep(seconds)                  # waiting, like a network request
    results.append(name)

def one_by_one(jobs):
    results = []
    for name, seconds in jobs:
        fake_download(name, seconds, results)
    return results

def with_threads(jobs):
    results = []
    threads = [threading.Thread(target=fake_download, args=(name, s, results)) for name, s in jobs]
    for t in threads:
        t.start()
    for t in threads:
        t.join()                         # wait for every thread to finish
    return results

jobs = [("a", 1.0), ("b", 0.5), ("c", 0.8)]
for run in (one_by_one, with_threads):
    start = time.perf_counter()
    order = run(jobs)
    print(run.__name__, order, round(time.perf_counter() - start, 1), "s")
```

And a process doing CPU-bound work in parallel (the `if __name__ == "__main__":` guard is required with `multiprocessing` on Windows and macOS):

```
import multiprocessing, os

def busy(n):
    return sum(i * i for i in range(n)), os.getpid()

if __name__ == "__main__":
    with multiprocessing.Pool() as pool:
        for total, pid in pool.map(busy, [3_000_000] * 4):
            print(total, "computed in process", pid)
```

## Practice

## Quiz

## Daily challenge

## Before you go

Run the two "On your own computer" programs if you can, and watch your computer's activity monitor while the `multiprocessing` one runs: you should see several Python processes busy at once. Tomorrow: the `threading` module in detail, with a simulator you can run here.